# 02 — Transform

**Purpose:** Clean, align, and merge raw datasets into analysis-ready tables saved to `data/processed/`.

## Steps
1. Load parquet files from `data/raw/`
2. Normalise date indices to a common monthly frequency
3. Compute derived series: yield curve spread (10y-2y), YoY inflation, real rates
4. Merge FRED, World Bank, and IMF into a unified panel
5. Forward-fill short gaps (≤2 months); flag longer gaps

## Outputs
- `data/processed/macro_panel.parquet` — unified monthly panel
- `data/processed/us_series.parquet` — US-only high-frequency series

## Papermill Parameters
- `run_date` — ISO date string injected by the GitHub Actions workflow

In [ ]:
run_date = None

In [ ]:
# Mount Google Drive for persistent storage (Colab only)
try:
    from google.colab import drive
    from pathlib import Path
    drive.mount("/content/drive")
    DRIVE_DATA = Path("/content/drive/MyDrive/macro-dashboard/data")
    _IN_COLAB = True
    print("Drive mounted. Reading from:", DRIVE_DATA)
except Exception:
    _IN_COLAB = False
    print("Not in Colab — using local data/ directory.")

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "pandas", "pyarrow", "numpy"])
print("Packages ready.")

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_DIR       = DRIVE_DATA / "raw"       if _IN_COLAB else Path("data/raw")
PROCESSED_DIR = DRIVE_DATA / "processed" if _IN_COLAB else Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
print(f"RAW_DIR       : {RAW_DIR}")
print(f"PROCESSED_DIR : {PROCESSED_DIR}")

In [ ]:
# --- Load raw parquets ---
fred_raw = pd.read_parquet(RAW_DIR / "fred_series.parquet")

try:
    wb_raw = pd.read_parquet(RAW_DIR / "worldbank.parquet")
    print("WB:    ", wb_raw.shape, "| index:", wb_raw.index.names)
except FileNotFoundError:
    print("WARNING: worldbank.parquet not found — World Bank data will be skipped")
    wb_raw = pd.DataFrame()

try:
    imf_raw = pd.read_parquet(RAW_DIR / "imf_weo.parquet")
    print("IMF:   ", imf_raw.shape, "| columns:", imf_raw.columns.tolist())
except FileNotFoundError:
    print("WARNING: imf_weo.parquet not found — IMF data will be skipped")
    imf_raw = pd.DataFrame()

try:
    oecd_raw = pd.read_parquet(RAW_DIR / "oecd_country.parquet")
    print("OECD:  ", oecd_raw.shape, "| latest:", oecd_raw.index.get_level_values("date").max().date())
except FileNotFoundError:
    print("NOTE: oecd_country.parquet not found — run 01_ingest first; OECD data skipped")
    oecd_raw = pd.DataFrame()

print("FRED:  ", fred_raw.shape, "| index:", fred_raw.index.dtype)


In [ ]:
# --- FRED: resample to month-end, derive indicators ---
fred = fred_raw.copy()
fred.index = pd.to_datetime(fred.index)
fred = fred.resample("ME").last()

# Yield curve spreads
fred["yield_spread_10y2y"] = fred["t10y"] - fred["t2y"]
fred["yield_spread_10y3m"] = fred["t10y"] - fred["t3m"]

# YoY inflation from index levels
fred["cpi_yoy_pct"] = fred["cpi_yoy"].pct_change(12) * 100
fred["pce_yoy_pct"] = fred["pce_yoy"].pct_change(12) * 100

# Approximate real 10y rate
fred["real_rate_10y"] = fred["t10y"] - fred["cpi_yoy_pct"]

# M2 YoY growth
fred["m2_yoy_pct"] = fred["m2"].pct_change(12) * 100

# Forward-fill short gaps only (≤2 months)
fred = fred.ffill(limit=2)

print("FRED monthly shape:", fred.shape)
print(fred[["yield_spread_10y2y", "cpi_yoy_pct", "real_rate_10y"]].tail(5))

In [ ]:
# --- World Bank: normalize to annual datetime index ---
if not wb_raw.empty:
    wb = wb_raw.reset_index()
    wb["date"] = pd.to_datetime(wb["date"].astype(str).str[:4], format="%Y")
    wb = wb.set_index(["country", "date"]).sort_index()
    wb.columns = ["wb_" + c for c in wb.columns]
    print("WB shape:", wb.shape)
else:
    wb = pd.DataFrame()
    print("World Bank data unavailable — skipping")

# --- IMF WEO: pivot to wide format ---
if not imf_raw.empty:
    imf = imf_raw.copy()
    imf["date"] = pd.to_datetime(imf["TIME_PERIOD"].astype(str), format="%Y")
    imf_wide = imf.pivot_table(
        index=["REF_AREA_LABEL", "date"],
        columns="CONCEPT_CODE",
        values="OBS_VALUE",
        aggfunc="first"
    ).rename_axis(index={"REF_AREA_LABEL": "country"})
    imf_wide.columns = ["imf_" + c for c in imf_wide.columns]
    # Map IMF WEO area labels to the display names the scoreboard queries by
    # (specs/006). Only the mismatches need an entry; the rest already match.
    IMF_LABEL_TO_DISPLAY = {
        "Türkiye, Republic of": "Turkey",
        "Poland, Republic of": "Poland",
        "Taiwan Province of China": "Taiwan",
        "Egypt, Arab Republic of": "Egypt",
        "Korea, Republic of": "South Korea",
    }
    imf_wide = imf_wide.rename(index=IMF_LABEL_TO_DISPLAY, level="country")
    print("IMF wide shape:", imf_wide.shape)
else:
    imf_wide = pd.DataFrame()
    print("IMF data unavailable — skipping")

# --- OECD: already wide (country, date) with oecd_* columns ---
oecd = oecd_raw.copy() if not oecd_raw.empty else pd.DataFrame()
if not oecd.empty:
    print("OECD wide shape:", oecd.shape)

# --- Merge WB + IMF + OECD into global panel ---
frames = [f for f in [wb, imf_wide, oecd] if not f.empty]
if not frames:
    global_panel = pd.DataFrame()
    print("WARNING: No global panel data — country scoreboard will be empty")
else:
    global_panel = frames[0]
    for other in frames[1:]:
        global_panel = global_panel.join(other, how="outer")
    global_panel = global_panel.sort_index().ffill(limit=2)
    print("Global panel shape:", global_panel.shape)
    if "United States" in global_panel.index.get_level_values("country"):
        us_rows = global_panel.xs("United States", level="country")
        show_cols = [c for c in global_panel.columns if c.startswith("oecd_") or c in ("imf_NGDP_RPCH", "imf_PCPIPCH", "imf_LUR")]
        print(us_rows[show_cols].dropna(how="all").tail(5))


In [ ]:
# --- Save outputs ---
fred.to_parquet(PROCESSED_DIR / "us_series.parquet")
print(f"US series saved:    {fred.shape} → {PROCESSED_DIR}/us_series.parquet")

global_panel.to_parquet(PROCESSED_DIR / "macro_panel.parquet")
print(f"Global panel saved: {global_panel.shape} → {PROCESSED_DIR}/macro_panel.parquet")